<a href="https://colab.research.google.com/github/PyGhosh/Grp_CH4124/blob/main/Group-A/Home-Assignment-3/230122020_HA_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayes and Naive Bayes: home assignment




In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
np.set_printoptions(precision=5, suppress=True)
print("Ready! Class order in manual examples: Insulator, Metal.")

Ready! Class order in manual examples: Insulator, Metal.


## Problem 1. Conditional probability

A new teaching dataset has rows [perovskite, not perovskite] and columns [oxide, not oxide]. Calculate the joint probability and both conditional probabilities.



In [2]:
counts = np.array([[4, 2], [3, 6]])
total = counts.sum()
p_joint = counts[0,0]/ total  # TODO: P(perovskite and oxide)
p_p_given_o = counts[0,0] /counts[:,0].sum()  # TODO: P(perovskite | oxide)
p_o_given_p = counts[0,0] / counts[0, : ].sum()  # TODO: P(oxide | perovskite)

if any(v is None for v in [p_joint, p_p_given_o, p_o_given_p]):
    print('Complete the three TODOs.')
else:
    assert np.allclose([p_joint, p_p_given_o, p_o_given_p], [4/15, 4/7, 4/6])
    print('Check passed:', p_joint, p_p_given_o, p_o_given_p)

Check passed: 0.26666666666666666 0.5714285714285714 0.6666666666666666


**Explain (1–2 sentences):** Why do the two conditional probabilities differ?

**Answer:** These probabilities condition on different events, so their denominators differ. Specifically, P(perovskite | oxide) uses all oxide cases, while P(oxide | perovskite) uses all perovskite cases.

## Problem 2. One positive screening result

A stable material occurs with prior probability 0.10. A filter flags 80% of stable and 20% of unstable materials. Compute P(flag) and P(stable | flag).



In [3]:
prior, sensitivity, fpr = 0.10, 0.80, 0.20
p_flag = sensitivity * prior + fpr * (1 - prior) # TODO
posterior = sensitivity*prior / p_flag  # TODO

if p_flag is None or posterior is None:
    print('Complete both TODOs.')
else:
    assert np.isclose(p_flag, 0.26)
    assert np.isclose(posterior*p_flag, sensitivity*prior)
    print('Check passed. Posterior:', posterior)

Check passed. Posterior: 0.30769230769230776


**Explain (1–2 sentences):** Why is the answer different from the sensitivity of 0.80?

**Answer:**  Sensitivity is P(flag | stable), whereas the requested posterior is P(stable | flag). The posterior also depends on how common stable materials are and on false positives among unstable materials.

## Problem 3. A second independent filter

Use the same rates as Problem 2. Write a reusable Bayes update function; apply it twice starting at 0.10. Assume the filters are independent conditional on the true class.


In [4]:
def update(prior, sensitivity, fpr):
    p_positive = sensitivity * prior + fpr * (1 - prior)
    posterior = sensitivity*prior / p_positive
    return posterior  # TODO

first = update(0.10, 0.80, 0.20)
second = update(first, 0.80, 0.20) # TODO: call update using first, after implementing the function

if first is None or second is None:
    print('Complete the function and second update.')
else:
    assert np.isclose(first, 4/13)
    assert np.isclose(second, 0.64)
    print('Check passed:', first, second)

Check passed: 0.30769230769230776 0.64


**Explain (1–2 sentences):** Would applying the same deterministic filter twice justify this update?

**Answer:** Not necessarily. Repeating the same deterministic filter on the same unchanged sample does not provide a new independent observation; the update is justified only if the second result supplies conditionally independent evidence with the stated rates.

## Problem 4. Beta update

Start with Beta(3,7). Observe 4 stable candidates in 12 trials. Calculate the posterior parameters and mean.


In [5]:
a, b, k, n = 3, 7, 4, 12
a_post = a + k  # TODO
b_post = b +(n-k) # TODO
post_mean = a_post / (a_post + b_post)  # TODO

if any(v is None for v in [a_post, b_post, post_mean]):
    print('Complete all TODOs.')
else:
    assert a_post == 7 and b_post == 15
    assert np.isclose(post_mean, a_post/(a_post+b_post))
    print('Check passed:', a_post, b_post, post_mean)

Check passed: 7 15 0.3181818181818182


**Explain (1–2 sentences):** Compare the posterior mean with the prior mean and the observed success fraction.

**Answer:** The prior mean is 3/(3+7) = 0.30, while the posterior mean is 7/(7+15) ≈ 0.318. The observed success fraction is 4/12 ≈ 0.333, so the posterior mean moves toward the observed fraction but remains moderated by the prior.

## Problem 5. Discrete classification
Classify a non-oxide with coordination number 6 and density below 5 g/cm³. Rows are [Insulator, Metal]; columns are [oxide, CN=6, low density]. Complement only the oxide column.


In [6]:
priors = np.array([2/3, 1/3])
base = np.array([[0.8, 0.5, 0.6], [0.3, 0.3, 0.4]])
observed = base.copy()
observed[:, 0] = 1 - observed[:, 0]  # non-oxide: complement oxide probabilities
scores = priors * np.prod(observed, axis=1)
posterior = scores / scores.sum()

if any(v is None for v in [observed, scores, posterior]):
    print('Complete all TODOs.')
else:
    assert np.allclose(scores, [0.04, 0.028])
    assert np.allclose(posterior, scores/scores.sum())
    print('Check passed. Posterior [I,M]:', posterior)

Check passed. Posterior [I,M]: [0.58824 0.41176]


**Explain (1–2 sentences):** Which class wins, and why is the decision less decisive than for an oxide?

**Answer:** The insulator class wins because its posterior score (0.040) is greater than the metal score (0.028). The result is less decisive because the non-oxide likelihoods are complements of the oxide likelihoods, and the resulting scores are relatively close.

## Problem 6. Laplace smoothing
Of 20 metals, zero have a binary feature. Of 40 insulators, eight have it. Compute smoothed P(feature=yes | class) for each class.



In [7]:
p_yes_metal = (0 + 1) / (20 + 2)
p_yes_insulator = (8 + 1) / (40 + 2)

if p_yes_metal is None or p_yes_insulator is None:
    print('Complete both TODOs.')
else:
    assert np.allclose([p_yes_metal,p_yes_insulator], [1/22,9/42])
    print('Check passed:', p_yes_metal, p_yes_insulator)

Check passed: 0.045454545454545456 0.21428571428571427


**Explain (1–2 sentences):** Why does an observed count of zero not establish physical impossibility?

**Answer:**  A zero count in a finite sample may reflect limited data rather than a true zero probability. Laplace smoothing adds a small pseudocount so the estimated probability remains nonzero and can generalize more sensibly.

## Problem 7. Gaussian density

Write the Gaussian density formula. Evaluate x=0.7 for a metal atomic-size descriptor with mean 0.6 Å and standard deviation 0.2 Å. Also calculate the z-score.



In [8]:
def normal_density(x, mu, sigma):
     return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))  # TODO
z = (0.7 - 0.6) / 0.2
density = normal_density(0.7, 0.6, 0.2)

if z is None or density is None:
    print('Complete both TODOs.')
else:
    assert np.isclose(z, 0.5)
    assert np.isclose(density, 1.7603266338)
    print('Check passed:', z, density)

Check passed: 0.4999999999999999 1.7603266338214976


**Explain (1–2 sentences):** Why is this density allowed to exceed one?

**Answer:** A probability density is not itself a probability; probability is obtained by integrating density over an interval. A narrow distribution can have density greater than one while its total area remains exactly one.

## Problem 8. Gaussian Naive Bayes

Classify x=[5.0,0.7,0.6]. Features are density, atomic size and electronegativity difference. Calculate the 2×3 density array, class scores, and normalized probabilities.



In [9]:
x = np.array([5.0, 0.7, 0.6])
means = np.array([[4, 1.3, 2.1], [4, 0.6, 0.4]])
stds = np.array([[2, 0.2, 0.4], [2, 0.2, 0.3]])
priors = np.array([2/3, 1/3])

heights = np.exp(-0.5 * ((x - means) / stds) ** 2) / (stds * np.sqrt(2 * np.pi))
scores = priors * np.prod(heights, axis=1)
posterior = scores / scores.sum()

if any(v is None for v in [heights, scores, posterior]):
    print('Complete all TODOs.')
else:
    assert np.shape(heights) == (2,3)
    reference = np.exp(-0.5*((x-means)/stds)**2)/(stds*np.sqrt(2*np.pi))
    assert np.allclose(heights, reference)
    assert np.allclose(scores, priors*np.prod(reference, axis=1))
    assert np.allclose(posterior, scores/scores.sum())
    print('Check passed. Posterior [I,M]:', posterior)

Check passed. Posterior [I,M]: [0.00002 0.99998]


**Explain (1–2 sentences):** Why does density provide no discrimination here?
Should the predicted probability be treated as experimentally established confidence?

**Answer:** Both classes have the same density mean (4) and standard deviation (2), so the density feature contributes the same Gaussian likelihood to both classes and does not change their relative scores. The resulting probability is a model-based estimate under the assumptions and parameters given, not experimentally established confidence.